# Explainability (SHAP) & Deterministic PSI Drift Simulation - Project 1
> **Production Alignment Note:** Uses the registered `Production` model (`fraud_detection_xgb`) for `shap.TreeExplainer` marginal attributions and a seeded random generator (`np.random.default_rng(42)`) for reproducible Population Stability Index (PSI) drift checks.

In [ ]:
import sys
from pathlib import Path
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
import shap
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.ingestion.utils import CANONICAL_FEATURE_COLS
from src.monitoring.observability import calculate_psi, evaluate_drift_status

DB_PATH = PROJECT_ROOT.parent / "mlflow_db" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
client = MlflowClient()
versions = client.search_model_versions("name='fraud_detection_xgb'")
prod_versions = [v for v in versions if v.current_stage == "Production"]
if not prod_versions:
    raise RuntimeError("No Production version found for fraud_detection_xgb")
prod_mv = max(prod_versions, key=lambda v: int(v.version))
model = mlflow.xgboost.load_model(f"models:/fraud_detection_xgb/{prod_mv.version}")
test_df = pd.read_parquet(PROJECT_ROOT / "data/processed/test.parquet")
X_sample = test_df[CANONICAL_FEATURE_COLS].iloc[:500]

In [ ]:
# Compute SHAP values on held-out sample using the loaded Production model
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0), index=CANONICAL_FEATURE_COLS
).sort_values(ascending=False)
print("Top 5 SHAP Features:\n", mean_abs_shap.head(5))

In [ ]:
# Deterministic PSI drift simulation (seeded with default_rng(42))
rng = np.random.default_rng(42)
baseline_v14 = test_df["V14"].to_numpy()
drifted_v14 = baseline_v14 + rng.normal(loc=1.5, scale=0.5, size=len(baseline_v14))
psi_score = calculate_psi(baseline_v14, drifted_v14)
print(
    f"Simulated V14 PSI: {psi_score:.4f} -> Status: {evaluate_drift_status(psi_score)}"
)